In [ ]:
from IPython.core.interactiveshell import InteractiveShell

InteractiveShell.ast_node_interactivity = "all"
import sympy as sym

from mixing_rotation.symbolic.general import (
    COUPLING,
    MASS,
    create_3d_rotation,
    create_4d_rotation,
)
from mixing_rotation.symbolic.pseudo import (
    create_mix_equation,
    rotated_pseudo_meson_couplings,
    bare_pseudo_masses,
    rotated_pseudo_photon_coupling,
    bare_pseudo_kaon_couplings,
    rotated_pseudo_kaon_couplings,
)

## Mixing Equation

In [ ]:
rotate, rotations, rotate_2d = create_4d_rotation()
pseudo_equation, pseudo_matrices = create_mix_equation(rotate)
pseudo_matrices = [rotate] + list(pseudo_matrices)

In [ ]:
# Rotation
print("The 2D rotation matrix is:")
sym.Eq(sym.MatrixSymbol(r"R_{2D}", 2, 2), rotate_2d, evaluate=False)
print(f"Which is used to create {len(rotations)} different 3D rotations:")
rotation_symbols = []
for rot in rotations:
    s = sym.Symbol(f"R_{{{list(rot.free_symbols)[0]}}}")
    sym.Eq(s, rot, evaluate=False)
    rotation_symbols.append(s)
print("And their multiplication creates the general 3D rotation matrix:")
rotate_symbol = sym.MatrixSymbol(r"R_{\tilde{\phi}}", rotate.rows, rotate.cols)
sym.Eq(
    rotate_symbol,
    sym.MatMul(*rotation_symbols, evaluate=False),
    evaluate=False,
)
sym.Eq(rotate_symbol, sym.MatMul(*rotations, evaluate=False), evaluate=False)

In [ ]:
matrix_symbols = [rotate_symbol]
for name in [f"{{{rotate_symbol}}}^\\text{{T}}", r"M_b", "M"]:
    matrix_symbols.append(sym.MatrixSymbol(name, rotate.rows, rotate.cols))
print("So our four matrices are:")
for s, mat in zip(matrix_symbols, pseudo_matrices):
    sym.Eq(s, mat, evaluate=False)
print("Which are all used in the mixing equation:")
sym.Eq(
    matrix_symbols[2],
    sym.MatMul(rotate_symbol, *matrix_symbols[1::-1]),
    evaluate=False,
)
pseudo_equation
sym.Eq(
    pseudo_equation.lhs,
    1 * pseudo_equation.rhs,
    evaluate=False,
)

In [ ]:
print("The following are the off-diagonal bare mass terms:")
for s, g in bare_pseudo_masses().items():
    sym.Eq(s, g, evaluate=False)

## Couplings

In [ ]:
print("The following are the rotated dark matter to meson couplings:")
for s, g in rotated_pseudo_meson_couplings(rotate.rows).items():
    sym.Eq(s, g, evaluate=False)

In [ ]:
print(
    "The following is the rotated pseudo scalar mediator to photon coupling (having factored out the un-rotated bare pion to photon and eta to photon couplings):"
)
sym.Eq(
    COUPLING["phi-gamma"], rotated_pseudo_photon_coupling(rotate.rows), evaluate=False
)

In [ ]:
print(
    "The following are the un-rotated bare pion-pion-kaon and eta-pion-kaon couplings (mediator-pion-kaon is zero):"
)
for s, g in bare_pseudo_kaon_couplings().items():
    sym.Eq(s, g, evaluate=False)

In [ ]:
print("The following are the rotated kaon couplings:")
for s, g in rotated_pseudo_kaon_couplings(rotate.rows).items():
    sym.Eq(s, g, evaluate=False)

In [ ]:
bares = bare_pseudo_kaon_couplings()
print("Substituting the un-rotated couplings into the rotated kaon couplings:")
for s, g in rotated_pseudo_kaon_couplings(rotate.rows).items():
    subbed = g.subs(bares)
    sym.Eq(s, subbed, evaluate=False)